# HW3 Part 2




**Course Code:**

**Group number:**

**Student Name:**

**Student ID:**

# Turn Continuity Classification


**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |


## 0. Environment Setup & Data Loading

In [3]:
!pip install datasets xgboost imbalanced-learn nltk -q


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix

# Advanced
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# NLP
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)


print("All imports successful!")

All imports successful!


In [5]:
# Load dataset (ensure train.csv and test.csv are in your current directory)
train_path = 'train.csv'
test_path = 'test.csv'

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

# Basic exploration
print(f"Train size: {len(train_df)}")
print("Label Distribution:\n", train_df['label'].value_counts())
display(train_df.head())

Train size: 1302
Label Distribution:
 label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [6]:
# ── Standardize column names ───────────────────────────────────────────────────

# Based on your files, the text column is named 'content'
# We will normalize it to 'text' for consistency in the pipeline
text_col_mapping = {'content': 'text'}

# Rename 'content' to 'text' if it exists
train_df = train_df.rename(columns=text_col_mapping)
public_test_df = public_test_df.rename(columns=text_col_mapping)

# Ensure 'id' column exists (if not already present)
if 'id' not in train_df.columns:
    train_df['id'] = train_df.index
if 'id' not in public_test_df.columns:
    public_test_df['id'] = public_test_df.index

# ── Label Analysis (Training Set Only) ────────────────────────────────────────

print("Standardization complete.")
print(f"Final columns: {train_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print("\nLabel distribution (train):")
    print(counts)

    # Calculate imbalance ratio if both classes 0 and 1 exist
    if 0 in counts and 1 in counts:
        ratio = counts[0] / counts[1]
        print(f"\nImbalance ratio (0/1): {ratio:.2f}")
    else:
        print("\nNote: Only one class found in 'label' column.")
else:
    print("\nWarning: 'label' column not found in train_df.")

# Note: public_test_df does not have a 'label' column, skipping its analysis.

Standardization complete.
Final columns: ['id', 'text', 'label']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64

Imbalance ratio (0/1): 0.56


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

In [8]:
# -----------------------------------------------------------------
# PHASE 1: Data Balancing
# -----------------------------------------------------------------

def perform_balancing(df, method='random'):
    """
    REQUIRED: method='random' (Random Over-sampling)
    OPTIONAL: method='advanced' (SMOTE)
    """
    if method == 'random':
        # Step 1: 分割多數類與少數類
        # 資料集中 label=1 有 837 筆（多數類），label=0 有 465 筆（少數類）
        
        # 篩選出 label=1 的所有資料列，存為多數類
        df_majority = df[df['label'] == 1]
        # 篩選出 label=0 的所有資料列，存為少數類
        df_minority = df[df['label'] == 0]

        # Step 2: 對少數類進行 Random Over-sampling
        df_minority_upsampled = resample(
            df_minority,                 # 要過取樣的資料（少數類 label=0）
            replace=True,                # 允許重複抽樣（放回抽樣），讓資料量可以增加
            n_samples=len(df_majority),  # 目標筆數 = 多數類的數量（837 筆）
            random_state=42              # 固定隨機種子，確保每次執行結果一樣
        )

        # Step 3: 合併兩類資料，並打亂順序
        # pd.concat 將多數類和過取樣後的少數類垂直合併成一個 DataFrame（共 1674 筆）
        df_balanced = pd.concat([df_majority, df_minority_upsampled])

        # sample(frac=1)：隨機打亂所有資料的順序（frac=1 代表取全部資料）
        # reset_index(drop=True)：重新編排 index 從 0 開始，drop=True 丟棄舊的 index
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

        # 回傳平衡後的 DataFrame，label=0 和 label=1 各 837 筆
        return df_balanced

    elif method == 'advanced':
        # SMOTE：合成少數類過取樣技術
        # SMOTE 需要數字向量才能計算樣本間距離，無法直接處理文字
        # 所以要先用 TF-IDF 將文字轉成數字向量

        # 建立 TF-IDF 向量化器
        # ngram_range=(1,1)：只看單字（unigram），不考慮片語組合
        # stop_words='english'：過濾掉 the, is, a 等無意義的英文常見字
        vectorizer = TfidfVectorizer(ngram_range=(1,1), stop_words='english')

        # fit_transform：先學習字典和每個字的 IDF 值（fit），再將文字轉成數字向量（transform）
        # x 的每一行代表一句話，每一列代表一個字的 TF-IDF 分數
        x = vectorizer.fit_transform(df['text'])

        # 取出 label 欄位作為目標變數 y
        y = df['label']

        # 建立 SMOTE 物件，random_state=42 固定隨機種子
        # k_neighbors=10 表示每個少數類樣本會找 10 個最近鄰來合成新樣本
        smote = SMOTE(random_state=42, k_neighbors=10)

        # fit_resample：SMOTE 找到少數類各樣本的 k 個最近鄰，
        # 在這些鄰近樣本之間隨機插值，合成全新的少數類樣本
        # 讓 label=0 從 465 筆增加到 837 筆，與 label=1 數量相同
        # 回傳 x_resampled（平衡後的數字向量）和 y_resampled（對應的 label）
        x_resampled, y_resampled = smote.fit_resample(x, y)

        # 同時回傳 vectorizer，讓外部驗證集可以用同一個字典和 IDF 進行轉換
        # 確保訓練集和驗證集的向量格式完全一致
        return x_resampled, y_resampled, vectorizer

# 驗證 Random Over-sampling 結果
sample_balanced = perform_balancing(train_df, method='random')
print("After Random Over-sampling:")
print(sample_balanced['label'].value_counts())  # 預期：label 0 和 1 各 837 筆

# 驗證 SMOTE 結果
# SMOTE 回傳三個值，需用三個變數分別接收
X_balanced, y_balanced, vectorizer_balanced = perform_balancing(train_df.copy(), method='advanced')
print("After SMOTE:")
print(pd.Series(y_balanced).value_counts())  # 預期：label 0 和 1 各 837 筆

After Random Over-sampling:
label
0    837
1    837
Name: count, dtype: int64
After SMOTE:
label
1    837
0    837
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------------------------
# PHASE 2: Data Balancing Comparison

# 比較三種 Data Balancing 方法：使用相同的 TF-IDF + SVM 模型
# 唯一變數是平衡方法，這樣比較才公平
# 用 80% 的 train_data 做訓練 (1041筆)，20% 的 train_data 作為驗證的 val_data (261筆)
# 其中 80% 的 train_data 中 label=1 有 669 筆，label=0 有 372 筆
# ------------------------------------------------------------------------------

"""方法一：Random Over-sampling"""

# 呼叫 perform_balancing，用 random 方法平衡 train_data
# .copy() 避免修改到原始 train_data
balanced_df = perform_balancing(train_data.copy(), method='random')

# 建立 TF-IDF 向量化器（專屬於 Random Over-sampling 方法）
# ngram_range=(1,1)：只看單字，stop_words='english'：過濾無意義的常見字
vectorizer_random = TfidfVectorizer(ngram_range=(1,1), stop_words='english')

# fit_transform：對平衡後的訓練資料學習字典並轉成數字向量
X_train_random = vectorizer_random.fit_transform(balanced_df['text'])

# 取出平衡後訓練資料的 label
y_train_random = balanced_df['label']

# transform（不是 fit_transform）：用同一個字典和 IDF 轉換驗證集
# 不能重新 fit，否則 IDF 值會不同，導致向量格式不一致
X_val_random = vectorizer_random.transform(val_data['text'])

# 取出驗證集的真實 label（之後三種方法都共用這個）
y_val_random = val_data['label']

# 建立 SVM 模型：kernel='linear' 用直線分類，probability=True 輸出機率
clf_random = SVC(kernel='linear', probability=True, random_state=42)

# 用平衡後的訓練資料訓練 SVM
clf_random.fit(X_train_random, y_train_random)

# 對驗證集做預測，得到每筆資料的預測 label
y_pred_random = clf_random.predict(X_val_random)

# 計算 Macro-F1：對每個 class 分別算 F1 再取平均，兩個 class 權重相同
# y_val_random 是驗證集的真實 label，y_pred_random 是模型預測的 label
# 拿真實答案和預測答案互相比對，算出 Macro-F1 分數，存進 f1_random
f1_random = f1_score(y_val_random, y_pred_random, average='macro')
print(f"Random Over-sampling Macro-F1: {f1_random:.4f}")


"""方法二：SMOTE"""

# SMOTE 回傳三個值：數字向量、label、以及 vectorizer
# x_train_smote 已經是 TF-IDF 轉換後的數字向量（不是 DataFrame）
x_train_smote, y_train_smote, vectorizer_smote = perform_balancing(train_data.copy(), method='advanced')

# 用 perform_balancing 內部的 vectorizer 轉換驗證集
# 確保使用同一個字典和 IDF，向量格式才會一致
x_val_smote = vectorizer_smote.transform(val_data['text'])

# 建立與方法一相同設定的 SVM（確保公平比較）
clf_smote = SVC(kernel='linear', probability=True, random_state=42)

# 用 SMOTE 平衡後的數字向量訓練 SVM
clf_smote.fit(x_train_smote, y_train_smote)

# 對驗證集做預測
y_pred_smote = clf_smote.predict(x_val_smote)

# 計算 Macro-F1（與方法一共用同一個 y_val_random 作為真實 label）
f1_smote = f1_score(y_val_random, y_pred_smote, average='macro')
print(f"SMOTE Macro-F1: {f1_smote:.4f}")


"""方法三：Cost-Sensitive Learning"""

# 不改變資料，改為告訴模型少數類的錯誤代價更高
# 直接使用原始不平衡的 train_data，不做任何過取樣

# 建立 TF-IDF 向量化器（專屬於 Cost-Sensitive 方法）
vectorizer_cs = TfidfVectorizer(ngram_range=(1,1), stop_words='english')

# fit_transform：對原始不平衡的訓練資料學習字典並轉成數字向量
X_train_cs = vectorizer_cs.fit_transform(train_data['text'])

# 取出訓練資料的 label（此時 label=1 有 669 筆，label=0 有 372 筆，仍不平衡）
y_train_cs = train_data['label']

# 用同一個 vectorizer 轉換驗證集
x_val_cs = vectorizer_cs.transform(val_data['text'])

# Cost-Sensitive SVM：
# class_weight='balanced'：自動根據 label 比例加重少數類的懲罰：
    # 計算方式：label=0 權重 = 總樣本數 / (2 × 372)，label=1 權重 = 總樣本數 / (2 × 669)
    # label=0 的權重較高，模型猜錯 label=0 的代價更大
"""
train_data 裡：
    總樣本數 = 1041
    class 數量 = 2（只有 label 0 和 label 1）
    label 0 有 372 筆
    label 1 有 669 筆
"""
clf_cs = SVC(kernel='linear', probability=True, random_state=42, class_weight='balanced')

# 用原始不平衡資料訓練，但模型會自動補償少數類
clf_cs.fit(X_train_cs, y_train_cs)

# 對驗證集做預測
y_pred_cs = clf_cs.predict(x_val_cs)

# 計算 Macro-F1
f1_cs = f1_score(y_val_random, y_pred_cs, average='macro')
print(f"Cost-Sensitive Macro-F1: {f1_cs:.4f}")


"""三種方法總比較"""

print("\n=== 三種方法比較 ===")
print(f"Random Over-sampling : {f1_random:.4f}")
print(f"SMOTE                : {f1_smote:.4f}")
print(f"Cost-Sensitive       : {f1_cs:.4f}")

# max() 找出 Macro-F1 最高的方法
# key=lambda x: x[1] 表示比較每個 tuple 的第二個值（Macro-F1 分數）
best = max(
    ('Random Over-sampling', f1_random),
    ('SMOTE', f1_smote),
    ('Cost-Sensitive', f1_cs),
    key=lambda x: x[1]
)
print(f"\n最佳方法: {best[0]} ({best[1]:.4f})")

Random Over-sampling Macro-F1: 0.6732
SMOTE Macro-F1: 0.6897
Cost-Sensitive Macro-F1: 0.7032

=== 三種方法比較 ===
Random Over-sampling : 0.6732
SMOTE                : 0.6897
Cost-Sensitive       : 0.7032

最佳方法: Cost-Sensitive (0.7032)


In [13]:
"""最佳參數下的 Cost-Sensitive Learning"""
# Cost-Sensitive SVM：
# kernel='rbf'：使用徑向基函數，能處理非線性分類邊界
# C=100：懲罰強度，越大越嚴格，不允許誤分類
# gamma='auto'：控制每個樣本的影響範圍，auto = 1/n_features

# 用調參後的最佳參數重新跑 Cost-Sensitive
# 上面的比較都用 kernel='linear' 確保公平
# 單獨展示 Cost-Sensitive 在最佳參數下的表現

clf_cs_best = SVC(kernel='rbf', C=100, gamma='auto', probability=True, random_state=42, class_weight='balanced')
clf_cs_best.fit(X_train_cs, y_train_cs)
y_pred_cs_best = clf_cs_best.predict(x_val_cs)
f1_cs_best = f1_score(y_val_random, y_pred_cs_best, average='macro')
print(f"Cost-Sensitive (best parameters) Macro-F1: {f1_cs_best:.4f}")

Cost-Sensitive (best parameters) Macro-F1: 0.7050


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [ ]:
# -----------------------------------------------------------------
# PHASE  3: Text Preprocessing (Optional)
# -----------------------------------------------------------------
"""
lemmatizer =
stop_words =
"""
def preprocess_text(text):

    [OPTIONAL PHASE 3]: Preprocessing Function
    Hint: Use .str.lower(), or apply a lambda function for NLTK PorterStemmer/WordNetLemmatizer.


"""
# Apply to entire corpus
train_df['text_clean'] = train_df['text'].apply(preprocess_text)
public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)

print("Sample original :", train_df['text'].iloc[0])
print("Sample cleaned  :", train_df['text_clean'].iloc[0])
"""

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2  ─  Model
# ══════════════════════════════════════════════════════════════════════════════
def get_model(model_type='svm'):
    """
    Model Factory for Baseline and Advanced models.
    """
    if model_type == 'svm':
        # REQUIRED PHASE 2 Baseline
        # TODO : complete the function, you could add or change the parameters
        return SVC(kernel='', probability=True, random_state=42)

    elif model_type == 'advanced':
        # [OPTIONAL PHASE 3]
        # Hint: Initialize xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss').
        # Try tuning max_depth, n_estimators, and learning_rate.
        return None

In [10]:
# --- Step 1: Internal Validation Setup ---
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])


print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print("Train label distribution:")
print(train_data['label'].value_counts())

Train subset : 1041 samples
Val   subset : 261 samples
Train label distribution:
label
1    669
0    372
Name: count, dtype: int64


In [ ]:
# TODO: Run your Baseline experiment
# 1. Balance data
# 2. Extract features (TF-IDF)
# 3. Train SVM
# 4. Evaluate using Macro-F1 and Classification Report

print("Running Baseline Experiment...")

# Define MODEL_TYPE locally for the baseline run
# This ensures this cell can be run independently for the baseline, even if kNNUqchf_Fic hasn't been run.
MODEL_TYPE = 'svm'
BALANCE_METHOD = 'random'

# 1. Balance data
balanced_train_data = perform_balancing(train_data.copy(), method=BALANCE_METHOD)
print(f"Balanced train size: {len(balanced_train_data)}  |  label dist:")
print(balanced_train_data['label'].value_counts())

# 2. Extract features (TF-IDF)
# TODO : complete the function, you could add or change the parameters
vectorizer = TfidfVectorizer(ngram_range=(), stop_words='')
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
X_train = vectorizer.fit_transform(balanced_train_data['text'])
y_train = balanced_train_data['label']

# 3. Train SVM
clf = get_model(MODEL_TYPE)
clf.fit(X_train, y_train)
print(f"Baseline model ({MODEL_TYPE}) trained on {len(X_train)} balanced samples.")

# 4. Evaluate using Macro-F1 and Classification Report
# Prepare validation data
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
X_val = vectorizer.transform(val_data['text'])
y_val = val_data['label']

# Make predictions
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f"=== Baseline: {MODEL_TYPE}| Balancing: {BALANCE_METHOD} ===")
print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 Score: {macro_f1:.4f}")

## Part 3: Enhancement with K-Fold (Optional)

Improve your results with implement  K-FOLD Cross Validtaion if needed.

In [ ]:
# --- Step 1: Training Pipeline ---

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3  ─  Stratified K-Fold Cross-Validation
#               Tests whether the simple split was just a lucky draw.
# ══════════════════════════════════════════════════════════════════════════════

def run_kfold_cv(df, model_type='svm', balance_method='random', n_splits=5, use_custom_preprocessing=False):
    """
    Run Stratified K-Fold CV and return fold scores + mean/std and all y_true/y_pred pairs.

    Parameters
    ----------
    df            : full training DataFrame
    model_type    : type of model to use ('svm' or 'advanced')
    balance_method: method for data balancing ('random' or 'advanced')
    n_splits      : number of folds (default 5)
    use_custom_preprocessing : bool, if True, uses the 'text_clean' column (after preprocess_text),
                                    else uses the original 'text' column.
    """


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [ ]:
# TODO: Train final model on the entire training set

# 1. Balance data using the best method found (e.g., 'random')
# Note: This will use the entire train_df, which has 'text_clean' if preprocessing was applied.
final_balanced_train_df = perform_balancing(train_df.copy(), method='random')
print(f"Full balanced training size: {len(final_balanced_train_df)}")
print(final_balanced_train_df['label'].value_counts())

# 2. Extract features (TF-IDF) on the full balanced dataset using the preprocessed text
# Use the same TF-IDF parameters (ngram_range, stop_words) as determined during validation/K-Fold
final_vectorizer = TfidfVectorizer() #TODO: fill in the parameters
# Ensure to use 'text_clean' column here for consistency with preprocessing if applied
X_full_train = final_vectorizer.fit_transform(final_balanced_train_df['text'])
y_full_train = final_balanced_train_df['label']

# 3. Train final SVM model
# The model type and hyperparameters should be chosen based on K-Fold results if K-Fold applied
final_model = get_model('svm')
final_model.fit(X_full_train, y_full_train)
print(f"Final model (svm) trained on {len(X_full_train)} samples using preprocessed text.")

In [ ]:
# -----------------------------------------------------------------
# PHASE 4: Kaggle Submission
# -----------------------------------------------------------------

def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv'):
    """
    Generates the final CSV. Ensure test_df['text'] is processed the same way as training data.
    """
    # Ensure test data is preprocessed before vectorization, just like training data
    # If 'text_clean' exists from previous preprocessing, use it; otherwise, use 'text'
    X_test = vectorizer.transform(test_df[text])
    test_predictions = model.predict(X_test)

    submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
    submission.to_csv(output_name, index=False)

    # Display first few rows of submission for verification
    print(f"Saved → {output_name}")
    print("Prediction distribution:")
    print(submission['label'].value_counts())
    display(submission.head(10))

    print(f"Success: {output_name} ready for Kaggle upload.")

# Final Step:
generate_kaggle_submission(final_model, final_vectorizer, public_test_df)

In [ ]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")